# Kaggle — Dual-view Uncertainty Herding

Runs the controlled disagreement-UH baseline and the parallel DINO/CellViT UCoverage method. Each of five rounds asks each branch for `2A` proposals and selects the final `A` from their union. Start with one experiment if estimating runtime on a new GPU.

In [ ]:
from pathlib import Path
import os, subprocess, sys, time

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "tiendung"
CODAPATH = Path("/kaggle/working/codapath")
if (CODAPATH / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(CODAPATH), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(CODAPATH), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(CODAPATH), "pull", "--ff-only", "origin", REPO_BRANCH])
elif CODAPATH.exists():
    raise RuntimeError(f"{CODAPATH} exists but is not a Git repository")
else:
    subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(CODAPATH)])
os.chdir(CODAPATH)
sys.path.insert(0, str(CODAPATH))
actual_branch = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
commit_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert actual_branch == REPO_BRANCH, (actual_branch, REPO_BRANCH)
required_source = CODAPATH / "sampling/dual_view_uherding.py"
assert required_source.is_file(), (
    f"{required_source} is absent from remote branch {REPO_BRANCH}. "
    "Commit and push the dual-view implementation before running Kaggle."
)
print("repo:", CODAPATH, "| branch:", actual_branch, "| commit:", commit_sha)

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# ---- EDIT THIS CELL ----
DATASET = "pathmnist"  # pathmnist | histoset | skintissue
SEED = 42
DATA_ROOT_HINT = Path("/kaggle/input/datasets/cryandrrich/nckh2026")
FEATURE_DIR_HINT = DATA_ROOT_HINT / "features"
NUCLEUS_DIR_HINT = DATA_ROOT_HINT / "nucleus_features/nucleus_features"
OUTPUT_DIR = Path("/kaggle/working/checkpoints")

# Primary controlled matrix. Comment out entries for a one-run pilot.
EXPERIMENTS = [
    ("disagreement_uherding", {"cell_pooling": "mean"}),
    ("dual_view_uherding", {"cell_pooling": "mean", "uncertainty_mode": "branch_margin", "fusion_mode": "joint"}),
    ("dual_view_uherding", {"cell_pooling": "rff", "uncertainty_mode": "branch_margin", "fusion_mode": "joint"}),
    ("dual_view_uherding", {"cell_pooling": "mean", "uncertainty_mode": "disagreement", "fusion_mode": "joint"}),
]
# Fusion ablations: fusion_mode = rrf | borda | score | visual_then_cell | cell_then_visual
# Loss ablation: consistency_weight = 0.05 and consistency_mode = symmetric_js | visual_teacher | cell_teacher

In [ ]:
import json, shutil
import numpy as np
import yaml
import torch
from run import main

with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)
assert config["cumulative_budget"] == [25, 50, 75, 100, 125, 150, 175, 200]
assert torch.cuda.is_available(), "Attach a Kaggle GPU before running"
assert DATASET in config["datasets"]
for required_sampler in ("dual_view_uherding", "disagreement_uherding"):
    assert required_sampler in config["samplers"], f"Missing config: {required_sampler}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
free_gpu, total_gpu = torch.cuda.mem_get_info()
free_disk = shutil.disk_usage("/kaggle/working").free
print(f"GPU={torch.cuda.get_device_name(0)} free={free_gpu/2**30:.1f}/{total_gpu/2**30:.1f} GiB")
print(f"working disk free={free_disk/2**30:.1f} GiB")
assert free_gpu >= 4 * 2**30, "Need at least 4 GiB free GPU memory for the configured kernel cap"
assert free_disk >= 3 * 2**30, "Need at least 3 GiB free working storage"

In [ ]:
from identity import sample_order_fingerprint
from load_data import get_data_loaders, get_sample_ids
from nucleus.cache import load_nucleus_cache

SEARCH_ROOTS = [DATA_ROOT_HINT, Path("/kaggle/input")]

def find_parent(probe, hint=None, max_depth=4):
    probe = Path(probe)
    roots = ([Path(hint)] if hint else []) + SEARCH_ROOTS
    for root in roots:
        if not root.exists():
            continue
        for depth in range(max_depth + 1):
            pattern = "/".join(["*"] * depth + list(probe.parts))
            hits = sorted(root.glob(pattern))
            if hits:
                return hits[0].parents[len(probe.parts) - 1]
    return None

data_probes = {
    "pathmnist": "pathmnist_224.npz",
    "histoset": "HistoSet-5x14/HistoSet-5x14",
    "skintissue": "SkinTissue/SkinTissue/tiles",
}
data_parent = find_parent(data_probes[DATASET], DATA_ROOT_HINT)
assert data_parent is not None, f"Dataset not found: {DATASET}"
data_path = data_parent / data_probes[DATASET]
assert data_path.exists(), data_path

# Resolve the exact seeded sample order before accepting either cache.
train_loader, test_loader, _ = get_data_loaders(str(data_path), SEED, False)
train_ids = get_sample_ids(train_loader.dataset)
test_ids = get_sample_ids(test_loader.dataset)
train_fingerprint = sample_order_fingerprint(train_ids)
test_fingerprint = sample_order_fingerprint(test_ids)
num_train, num_test = len(train_ids), len(test_ids)
del train_loader, test_loader

nucleus_parent = find_parent(f"{DATASET}_seed{SEED}/manifest.json", NUCLEUS_DIR_HINT)
assert nucleus_parent is not None, "Aligned nucleus cache not found"
nucleus_path = nucleus_parent / f"{DATASET}_seed{SEED}"
nucleus_cache = load_nucleus_cache(str(nucleus_path), expected_sample_ids=train_ids)
assert nucleus_cache.manifest.get("dataset") == DATASET
assert nucleus_cache.manifest.get("seed") == SEED
assert nucleus_cache.cellvit_embeddings is not None, "CellViT embeddings absent from cache"
assert nucleus_cache.num_patches == num_train

# An invalid cache mounted under /kaggle/input is read-only. Validate all
# metadata here; otherwise re-extract into /kaggle/working instead.
vit_name = config.get("models", {}).get("vit", "facebook/dinov2-base")
safe_vit = vit_name.replace("/", "_")
feature_base = f"{DATASET}_seed{SEED}_{safe_vit}"
feature_parent = find_parent(f"{feature_base}_train.npy", FEATURE_DIR_HINT)
feature_valid = False
if feature_parent is not None:
    train_file = feature_parent / f"{feature_base}_train.npy"
    test_file = feature_parent / f"{feature_base}_test.npy"
    feature_manifest_file = feature_parent / f"{feature_base}_manifest.json"
    if train_file.is_file() and test_file.is_file() and feature_manifest_file.is_file():
        with open(feature_manifest_file, "r", encoding="utf-8") as handle:
            feature_manifest = json.load(handle)
        train_shape = np.load(train_file, mmap_mode="r").shape
        test_shape = np.load(test_file, mmap_mode="r").shape
        feature_valid = (
            feature_manifest.get("dataset") == DATASET
            and feature_manifest.get("seed") == SEED
            and feature_manifest.get("backbone") == vit_name
            and feature_manifest.get("train_fingerprint") == train_fingerprint
            and feature_manifest.get("test_fingerprint") == test_fingerprint
            and train_shape[0] == num_train and test_shape[0] == num_test
        )
if not feature_valid:
    feature_parent = Path("/kaggle/working/features")
    feature_parent.mkdir(parents=True, exist_ok=True)
    print("No exact aligned DINO cache; the first run will extract into working storage.")
print("data:", data_path, f"train={num_train} test={num_test}")
print("DINO cache:", feature_parent, "aligned=", feature_valid)
print("nucleus cache:", nucleus_path, f"cells={nucleus_cache.num_cells}")

In [ ]:
from nucleus.ragged import pool_ragged_rff
from sampling import get_sampler
from set_up import set_seed

# Resource contract for the precomputed relevant-set kernels. Two kernels
# are resident for dual-view joint fusion; one extra matrix is a conservative
# allowance for a temporary marginal-gain tensor.
dual_cfg = config["samplers"]["dual_view_uherding"]
candidate_cap = dual_cfg.get("candidate_pool_size")
assert candidate_cap is not None, "Do not use full-pool kernels on PathMNIST/T4"
max_relevant = int(candidate_cap) + max(config["cumulative_budget"])
one_kernel_gib = max_relevant * max_relevant * 4 / 2**30
estimated_kernel_peak = 3 * one_kernel_gib
free_gpu, _ = torch.cuda.mem_get_info()
assert estimated_kernel_peak * 2**30 + 2 * 2**30 < free_gpu, (
    f"Configured candidate cap needs roughly {estimated_kernel_peak:.2f} GiB "
    "for kernels plus 2 GiB headroom; lower candidate_pool_size."
)
print(f"candidate cap={candidate_cap}; estimated kernel working set≈{estimated_kernel_peak:.2f} GiB")

# Exercise RFF on real cached CellViT rows before starting any AL run.
pilot_patches = min(128, nucleus_cache.num_patches)
pilot_offsets = np.asarray(nucleus_cache.offsets[:pilot_patches + 1], dtype=np.int64)
pilot_cells = int(pilot_offsets[-1])
pilot_view = pool_ragged_rff(
    nucleus_cache.cellvit_embeddings[:pilot_cells], pilot_offsets,
    nucleus_cache.confidence[:pilot_cells], output_dim=16, seed=SEED,
)
assert pilot_view.patch_features.shape == (pilot_patches, 16)
assert np.isfinite(pilot_view.patch_features).all()
print("real-cache RFF pilot PASS:", pilot_view.metadata)

# GPU smoke covers both registered entry points and the exact five-round
# kernel/probe path with missing cell views.
rng = np.random.default_rng(SEED)
smoke_n = 30
smoke_common = dict(
    image_embeddings=rng.normal(size=(smoke_n, 12)).astype(np.float32),
    nucleus_embeddings=rng.normal(size=(smoke_n, 10)).astype(np.float32),
    nucleus_reliability=np.r_[np.zeros(2), np.ones(smoke_n - 2)].astype(np.float32),
    oracle_labels=np.arange(smoke_n) % 3, num_classes=3, max_budget=10,
    device=torch.device("cuda"), num_rounds=5, candidate_pool_size=None,
    chunk_size=30, n_sigma=30, probe_epochs=1, probe_lr=1e-3, diag=False,
)
for smoke_sampler, smoke_overrides in [
    ("disagreement_uherding", {}),
    ("dual_view_uherding", {"uncertainty_mode": "disagreement", "fusion_mode": "joint"}),
]:
    set_seed(SEED)
    smoke_selected = get_sampler(name=smoke_sampler, **smoke_common, **smoke_overrides)
    assert len(smoke_selected) == 10 and len(set(smoke_selected)) == 10
    print(smoke_sampler, "GPU smoke PASS")
del nucleus_cache, pilot_view
torch.cuda.empty_cache()

In [ ]:
training_cfg = config.get("training", {})
dataset_cfg = config["datasets"][DATASET]
for experiment_index, (sampler_name, overrides) in enumerate(EXPERIMENTS, start=1):
    sampler_cfg = {**config["samplers"][sampler_name], **overrides}
    print("=" * 80)
    print(f"EXPERIMENT {experiment_index}/{len(EXPERIMENTS)}: {sampler_name} {overrides}")
    started = time.time()
    main(
        data_path=str(data_path), sampler_name=sampler_name,
        num_classes=dataset_cfg["num_classes"],
        cumulative_budget=config["cumulative_budget"],
        data_descriptions=dataset_cfg["descriptions"],
        prompt_templates=config["prompt_templates"],
        sampler_cfg=sampler_cfg,
        probe_epochs=training_cfg["probe_epochs"],
        probe_lr=training_cfg["probe_lr"],
        device=torch.device(config["device"]), random_seed=SEED,
        save_dir=str(OUTPUT_DIR / DATASET), verbose=True,
        model_cfg=config.get("models", {}),
        feature_cache_dir=str(feature_parent),
        nucleus_cache_dir=str(nucleus_parent), run_name=None,
    )
    print(f"Finished in {(time.time() - started) / 60:.1f} minutes")

In [ ]:
def artifact_name(sampler_name, sampler_cfg):
    pooling = sampler_cfg.get("cell_pooling", "mean")
    if sampler_name == "disagreement_uherding":
        return f"disagreement_uh_{pooling}"
    uncertainty = sampler_cfg.get("uncertainty_mode", "branch_margin")
    fusion = sampler_cfg.get("fusion_mode", "joint")
    return f"dual_uh_{pooling}_{uncertainty}_{fusion}"

artifact_dir = OUTPUT_DIR / DATASET
expected_names = []
for sampler_name, overrides in EXPERIMENTS:
    sampler_cfg = {**config["samplers"][sampler_name], **overrides}
    name = artifact_name(sampler_name, sampler_cfg)
    expected_names.append(name)
    for budget in config["cumulative_budget"]:
        selected_path = artifact_dir / f"{name}_selected_budget_{budget}.pt"
        probe_path = artifact_dir / f"{name}_probe_budget_{budget}.pt"
        assert selected_path.is_file(), selected_path
        assert probe_path.is_file(), probe_path
        payload = torch.load(selected_path, map_location="cpu", weights_only=False)
        indices = payload["selected_indices"]
        assert len(indices) == budget and len(set(indices)) == budget
        assert min(indices) >= 0 and max(indices) < num_train
        assert payload["sampler"] == sampler_name
        assert payload["run_name"] == name
        assert payload["train_fingerprint"] == train_fingerprint
        pooling_meta = payload["nucleus_manifest"]["pooling"]
        assert pooling_meta["pooling"] == sampler_cfg.get("cell_pooling", "mean")
        if pooling_meta["pooling"] == "rff":
            assert np.isfinite(pooling_meta["rff_bandwidth"])
    results_path = artifact_dir / f"{name}_results.pt"
    assert results_path.is_file(), results_path
    results = torch.load(results_path, map_location="cpu", weights_only=False)
    assert results["budgets"] == config["cumulative_budget"]
    assert sorted(results["linear"]) == config["cumulative_budget"]
    print(name, "artifact audit PASS")
assert len(expected_names) == len(set(expected_names)), "Experiment artifact names collide"
archive = shutil.make_archive(
    "/kaggle/working/dual_view_uherding_results", "zip",
    root_dir=str(OUTPUT_DIR), base_dir=DATASET,
)
print("ALL ARTIFACTS PASS; archive:", archive)